# MeetStream Bridge + LlamaIndex — Real, Live Test Notebook

Runs a full real pipeline against the **real** MeetStream API with **real** credentials — a
real bot joins a real meeting, a real transcript is fetched, and the LlamaIndex reader,
tools, and agent run against that real data. No mocks, no placeholder bridge key, no
skip-if-blank guards.

**Before running**: fill in every `<INSERT ... HERE>` value in section 2 with your real
MeetStream API key and either a real meeting URL or an existing `bot_id`. Cells that need a
value you left blank will raise, not silently skip — that's deliberate, so a half-configured
run fails loudly instead of quietly passing on empty data.

Structure:
1. Install the local projects (editable, from local source — no PyPI involved)
2. Configuration — insert your real credentials and meeting info here
3. Start a real bridge server, pointed at your real MeetStream key
4. LlamaIndex: client, tools, dispatch, transcript, reader, agent — all real
5. RAG over the real transcript
6. Cleanup (stop the bridge)


## 1. Install the local projects (editable, from local source)

`pip install -e <path>` installs directly from the local directory — this repo has never
been published to PyPI, and none of these commands need it to be. `[dev,examples]` are
optional-dependency groups defined in each project's own `pyproject.toml`, not PyPI
packages.


In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "docs" / "ARCHITECTURE.md").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError(
            "Could not locate the repository root (expected to find docs/ARCHITECTURE.md). "
            "Run this notebook from inside the cloned repository."
        )
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT / "scripts" / "verification"))
import _lib

print("Repo root:", REPO_ROOT)
print("Python:", sys.executable)


Repo root: /Users/software/development/meetstream-ai/meetstream-langchain/llamaindex-repo
Python: /opt/homebrew/opt/python@3.11/bin/python3.11


In [2]:
!{sys.executable} -m pip install -e "{REPO_ROOT / 'bridge'}[dev]"
!{sys.executable} -m pip install -e "{REPO_ROOT}[dev,examples]"


Obtaining file:///Users/software/development/meetstream-ai/meetstream-langchain/llamaindex-repo/bridge
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for meetstream-bridge (pyproject.toml) ... done
  Created wheel for meetstream-bridge: filename=meetstream_bridge-0.1.0-py3-none-any.whl size=2013 sha256=112c46d8f2671be13d71608a46f0daab6a3b7d6237866e5432fc18abf3d17993
  Stored in directory: /private/var/folders/ck/gs22xnt513x04n1m13hg3w6h0000gr/T/pip-ephem-wheel-cache-8mya67nt/wheels/d6/51/67/7808fc5cc956fd875ff8c0ed3224f126514611f85342f35c03
Successfully built meetstream-bridge
  Attempting uninstall: meetstream-bridge
    Found existing installation: meetstream-bridge 0.1.0
    Uninstalling meetstream-bridge-0.1.0:
      Successfully uninstalled meetstream-bridge-0.

In [3]:
# Editable installs register their import hooks via a .pth file that Python's
# `site` module only processes at interpreter startup -- a package installed
# mid-session (like the one above) isn't importable in *this* kernel without
# either a restart or pointing sys.path directly at its source directory, which
# is what this cell does. The bridge subprocess started later doesn't hit this
# -- it's a fresh process that reads the now-complete install normally.
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


## 2. Configuration — insert your real credentials here

Replace every `<INSERT ... HERE>` below. Nothing here is committed anywhere — this cell only
sets local Python variables for the rest of this notebook.

- `MEETSTREAM_API_KEY` — **required**. Your real MeetStream key; the bridge sends this to
  MeetStream as `Authorization: Token <key>`.
- `MEETING_URL` — a real, currently-joinable Google Meet / Zoom / Teams URL. Use this to have
  the bot dispatched by this notebook join a live meeting.
- `EXISTING_BOT_ID` — alternative to `MEETING_URL`: a `bot_id` from a bot you already
  dispatched (via this notebook or `live_meetstream_test.py dispatch`) whose transcript has
  already finished processing. Fill in exactly one of `MEETING_URL` / `EXISTING_BOT_ID`.
- `OPENAI_API_KEY` — needed only for the agent and RAG cells (real LLM calls). Leave as the
  placeholder to skip just those cells; everything else still runs for real.


In [ ]:
MEETSTREAM_API_KEY = "xxxxxxx" # input

# Fill in exactly one of these two:
MEETING_URL = "https://meet.google.com/jsa-qvvo-rby"
EXISTING_BOT_ID = ""  # e.g. "bot_123" -- leave blank if using MEETING_URL instead

# Only needed for the agent/RAG cells -- leave as the placeholder to skip just those.
OPENAI_API_KEY = "xxxxxx" # input


def _is_placeholder(value: str) -> bool:
    return not value or value.startswith("<INSERT")


if _is_placeholder(MEETSTREAM_API_KEY):
    raise ValueError("Set MEETSTREAM_API_KEY above to your real MeetStream API key before continuing.")

if _is_placeholder(MEETING_URL) and not EXISTING_BOT_ID:
    raise ValueError("Set MEETING_URL to a real meeting URL, or EXISTING_BOT_ID to an already-dispatched bot_id.")
if not _is_placeholder(MEETING_URL) and EXISTING_BOT_ID:
    raise ValueError("Set only one of MEETING_URL / EXISTING_BOT_ID, not both.")
if _is_placeholder(MEETING_URL):
    MEETING_URL = ""  # normalize the unused placeholder to empty

RUN_LLM_CELLS = not _is_placeholder(OPENAI_API_KEY)
if RUN_LLM_CELLS:
    import os

    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY  # llama-index reads this from the environment
else:
    print("OPENAI_API_KEY left as placeholder -- agent and RAG cells below will be skipped.")


## 3. Start the bridge

Reuses `TemporaryBridge` from `scripts/verification/_lib.py` — the same helper the
verification suite uses to spin up a real `uvicorn app.main:app` subprocess and wait for
`/health`, pointed at your real `MEETSTREAM_API_KEY`. This calls `__enter__`/`__exit__`
manually (instead of a `with` block) so the bridge stays alive across the rest of this
notebook; it's stopped explicitly in the Cleanup section at the end.


In [5]:
bridge = _lib.TemporaryBridge(api_key=MEETSTREAM_API_KEY)
bridge.__enter__()
print("Bridge running at", bridge.base_url)
print("Health check:", _lib.bridge_is_running(bridge.base_url))


Bridge running at http://127.0.0.1:56642
Health check: True


## 4. LlamaIndex: client + tools + reader + agent


In [6]:
from llama_index_meetstream import MeetStreamAPIError, MeetStreamClient, MeetStreamReader
from llama_index_meetstream.tools import get_meetstream_tools

li_client = MeetStreamClient(api_key=MEETSTREAM_API_KEY, base_url=bridge.base_url)
li_tools = get_meetstream_tools(li_client)
print("LlamaIndex tools:", [t.metadata.name for t in li_tools])


LlamaIndex tools: ['dispatch_meetstream_bot', 'get_meeting', 'get_meeting_transcript', 'send_meeting_chat_message', 'send_meeting_image', 'leave_meeting']


### Dispatch a bot, or reuse an existing `bot_id`


In [7]:
if MEETING_URL:
    dispatch_result = li_client.dispatch_bot(meeting_url=MEETING_URL, bot_name="Notebook Test Bot")
    bot_id = dispatch_result.meeting_id
    print("Dispatched:", dispatch_result)
else:
    bot_id = EXISTING_BOT_ID
    print("Using existing bot_id:", bot_id)


Dispatched: bot_id='2a7fdad3-c0c1-4de6-8dba-fe2419eb39e7' meeting_id='2a7fdad3-c0c1-4de6-8dba-fe2419eb39e7' status='Active'


### Check meeting status


In [12]:
print(li_client.get_meeting(bot_id))


meeting_id='2a7fdad3-c0c1-4de6-8dba-fe2419eb39e7' status='Done' platform=None meeting_url=None title=None started_at=None ended_at=None custom_attributes=None


### Wait for and fetch the transcript

Transcription is asynchronous on MeetStream's side -- this polls `get_transcript`, treating
`transcript_not_ready` as "try again shortly" rather than a failure, same as
`live_llamaindex_test.py`. If you just dispatched a fresh bot, join the meeting and let it run
a bit before executing this cell, or it will spend its retries waiting.


In [13]:
import time


def wait_for_transcript(client, meeting_id, attempts=10, delay_seconds=15):
    for attempt in range(attempts):
        try:
            return client.get_transcript(meeting_id)
        except MeetStreamAPIError as exc:
            if exc.code != "transcript_not_ready":
                raise
            print(f"Not ready yet ({attempt + 1}/{attempts}) -- waiting {delay_seconds}s...")
            time.sleep(delay_seconds)
    raise TimeoutError(f"Transcript for {meeting_id} still not ready after {attempts} attempts")


transcript = wait_for_transcript(li_client, bot_id)
print(f"{len(transcript.segments)} real segment(s)")


1 real segment(s)


### Load as LlamaIndex `Document`s


In [14]:
li_docs = MeetStreamReader(meeting_id=bot_id, client=li_client).load_data()
for doc in li_docs[:3]:
    print(doc.text)
    print(doc.metadata)
    print()


Index and line chain wrappers for the midstream API key. Hello. This is a test for the line chain and line index wrappers for the midstream APIs.
{'source': 'meetstream', 'meeting_id': '2a7fdad3-c0c1-4de6-8dba-fe2419eb39e7', 'meeting_title': None, 'platform': None, 'speaker': 'Anush Somasundaram', 'start_time': 0.24, 'end_time': 12.16}



### Invoke a tool directly (no LLM)

Proves the tool itself works, independent of whether an LLM would have chosen to call it --
same idea as `live_llamaindex_tools.py`.


In [15]:
transcript_tool = next(t for t in li_tools if t.metadata.name == "get_meeting_transcript")
result = transcript_tool.call(meeting_id=bot_id)
print(f"{len(result.raw_output['segments'])} segment(s)")


1 segment(s)


### Run a real LlamaIndex agent

Same check as `scripts/verification/live_llamaindex_agent.py`: does a real tool-calling model
decide on its own to call `get_meeting_transcript`. Skipped if `OPENAI_API_KEY` was left as
the placeholder in section 2.


In [16]:
if RUN_LLM_CELLS:
    from llama_index.core.agent.workflow import FunctionAgent
    from llama_index.llms.openai import OpenAI as LlamaIndexOpenAI

    li_agent = FunctionAgent(tools=li_tools, llm=LlamaIndexOpenAI(model="gpt-4o-mini"))
    li_response = await li_agent.run(user_msg=f"What did people say in meeting {bot_id}? Give me a short summary.")
    print(li_response)
else:
    print("Skipping -- OPENAI_API_KEY not set in section 2.")


In the meeting with ID 2a7fdad3-c0c1-4de6-8dba-fe2419eb39e7, Anush Somasundaram spoke about "Index and line chain wrappers for the midstream API key." They mentioned that it was a test for the line chain and line index wrappers related to the midstream APIs. The discussion lasted from 0.24 to 12.16 seconds into the meeting.


## 5. RAG over the real transcript

Minimal inline version of `examples/rag_example.py` — reuses the `Document`s already loaded
above rather than re-fetching. Skipped if `OPENAI_API_KEY` was left as the placeholder in
section 2.


In [17]:
if RUN_LLM_CELLS:
    from llama_index.core import VectorStoreIndex

    index = VectorStoreIndex.from_documents(li_docs)
    print(index.as_query_engine().query("What was discussed in this meeting?"))
else:
    print("Skipping -- OPENAI_API_KEY not set in section 2.")


The meeting discussed the index and line chain wrappers for the midstream API key, specifically testing the line chain and line index wrappers for the midstream APIs.


## 6. Cleanup


In [18]:
bridge.__exit__(None, None, None)
print("Bridge stopped.")

Bridge stopped.
